# ⚡ Notebook 2: Database Optimization for Writes

Before adding complexity, exhaust what your existing database can do. Proper tuning can often 10x your write throughput.

## Learning Objectives

By the end of this notebook, you'll understand:
- Write-optimized database patterns
- Index overhead and management
- Bulk insert techniques
- When to choose specialized databases

---

🔍 **Open Adminer** at http://localhost:8080 to watch write performance!

In [ ]:
import psycopg2
import psycopg2.extras
import time
import statistics
from concurrent.futures import ThreadPoolExecutor

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "writes_demo",
    "user": "demo",
    "password": "demo"
}

def get_connection():
    return psycopg2.connect(**DB_CONFIG)

print("✅ Connected to PostgreSQL")

## 📚 Write-Optimized Database Patterns

In [ ]:
print("📚 Database Types by Write Pattern")
print("=" * 60)
print("""
TRADITIONAL RDBMS (PostgreSQL, MySQL)
─────────────────────────────────────────────────────────────
• Updates data IN PLACE (requires disk seeks)
• Maintains B-tree indexes (rebalancing overhead)
• Strong consistency, ACID transactions
• Good for: Mixed workloads, complex queries
• Writes: ~1,000-50,000/sec depending on tuning

LOG-STRUCTURED (Cassandra, RocksDB)
─────────────────────────────────────────────────────────────
• APPEND-ONLY writes (sequential, fast!)
• Compaction merges data in background
• Eventually consistent (tunable)
• Good for: Write-heavy, simple queries
• Writes: ~10,000-100,000/sec

TIME-SERIES (InfluxDB, TimescaleDB)
─────────────────────────────────────────────────────────────
• Optimized for timestamp-ordered data
• Compression for sequential values
• Automatic data retention/rollup
• Good for: Metrics, IoT, logs
• Writes: ~100,000+/sec

KEY-VALUE (Redis, DynamoDB)
─────────────────────────────────────────────────────────────
• Simple put/get operations
• In-memory or SSD-optimized
• Horizontal scaling built-in
• Good for: Counters, sessions, caches
• Writes: ~100,000+/sec
""")

## 🧮 Write Amplification: Putting a Number on "LSM is Faster"

The table above says Cassandra writes faster than PostgreSQL, which is the kind
of claim that is easy to repeat and hard to defend. The number that actually
explains it is **write amplification (WA)**: bytes written to disk per byte of
user data.

**B-tree (PostgreSQL, InnoDB).** Your 200-byte row lands inside an 8 KiB page,
and the page is the unit of I/O -- the whole page gets written, not your row.
Worse, it gets written twice: once into the WAL as a full-page image (the first
time a page is touched after a checkpoint) and once when the dirty page is
flushed. Every secondary index adds another page, in another *random* location.

**LSM tree (Cassandra, RocksDB).** A write goes into an in-memory memtable plus
a sequential commit-log append -- both cheap. The expensive part is deferred to
**compaction**, which rewrites each level. With leveled compaction, fanout `T`
and `L` levels, a byte gets rewritten roughly `T/2` times per level.

Let's do the arithmetic.

In [ ]:
print("🧮 Write Amplification Arithmetic")
print("=" * 60)

ROW_BYTES = 200
PAGE_BYTES = 8 * 1024


def btree_wa(num_indexes: int) -> float:
    """Bytes written per row for a B-tree engine.

    One heap page plus one page per secondary index -- each written to the WAL
    as a full-page image, then again when the page itself is flushed.
    """
    pages_touched = 1 + num_indexes
    wal_bytes = pages_touched * PAGE_BYTES
    data_bytes = pages_touched * PAGE_BYTES
    return (wal_bytes + data_bytes) / ROW_BYTES


def lsm_wa(levels: int = 5, fanout: int = 10) -> float:
    """Bytes written per row for a leveled-compaction LSM engine.

    1x for the commit log, 1x for the memtable flush into L0, then each of the
    remaining levels rewrites the data ~fanout/2 times on average.
    """
    return 1 + 1 + (levels - 1) * (fanout / 2)


print(f"\nAssumptions: {ROW_BYTES}-byte row, {PAGE_BYTES // 1024} KiB page\n")
print(f"{'engine':<40}{'write amplification':>20}")
print("-" * 60)
for n in (0, 3, 5):
    label = f"B-tree, {n} secondary index{'es' if n != 1 else ''}"
    print(f"{label:<40}{btree_wa(n):>19.0f}x")
print(f"{'LSM, leveled (L=5, T=10)':<40}{lsm_wa():>19.0f}x")
print(f"{'LSM, size-tiered (L=5, T=4)':<40}{lsm_wa(5, 4):>19.0f}x")

print(f"""
💡 Read those two families against each other:

   • With 3 secondary indexes the B-tree writes ~{btree_wa(3):.0f}x your row.
     The leveled LSM writes ~{lsm_wa():.0f}x.
   • But the *shape* matters more than the ratio. The B-tree's bytes are
     RANDOM page writes on the request path, paid synchronously at COMMIT.
     The LSM's bytes are SEQUENTIAL, paid asynchronously by compaction.

   On spinning disks random-vs-sequential was a ~100x difference and the LSM
   won by a mile. On NVMe it is more like 3-10x, which is exactly why the gap
   between the two families has narrowed and why "just use Cassandra" stopped
   being automatic advice.

   The LSM bill still comes due, just later: compaction competes with
   foreground traffic, and a read may have to check several levels (which is
   what bloom filters are there to avoid).
""")

# The whole argument rests on a well-indexed B-tree amplifying more than a
# leveled LSM. If that ever inverts, the prose above is wrong.
assert btree_wa(3) > lsm_wa(), (
    f"expected B-tree WA ({btree_wa(3):.0f}x) > leveled LSM WA ({lsm_wa():.0f}x)"
)
# Indexes are the dominant term for a B-tree: each one is another random page.
assert btree_wa(5) > btree_wa(0) * 2, (
    f"5 indexes should more than double B-tree WA vs a heap-only table, got "
    f"{btree_wa(5):.0f}x vs {btree_wa(0):.0f}x"
)


## 🔧 Index Management for Writes

In [ ]:
# Two structurally identical tables. The only difference is how many indexes
# PostgreSQL has to keep up to date on every INSERT -- which is the one variable
# the next cell measures.
conn = get_connection()
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS events_no_index (
        id SERIAL PRIMARY KEY,
        event_type VARCHAR(50),
        user_id INTEGER,
        payload JSONB,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
""")

cursor.execute("""
    CREATE TABLE IF NOT EXISTS events_with_indexes (
        id SERIAL PRIMARY KEY,
        event_type VARCHAR(50),
        user_id INTEGER,
        payload JSONB,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
""")

# Five extra indexes so the per-INSERT overhead is clearly visible.
# In the real world, tables with 5–10 indexes are very common.
cursor.execute("CREATE INDEX IF NOT EXISTS idx_ewi_type ON events_with_indexes(event_type)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_ewi_user ON events_with_indexes(user_id)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_ewi_time ON events_with_indexes(created_at)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_ewi_type_user ON events_with_indexes(event_type, user_id)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_ewi_payload ON events_with_indexes USING GIN (payload)")

cursor.execute("TRUNCATE events_no_index, events_with_indexes")
conn.commit()
conn.close()

print("✅ Test tables created")
print("   • events_no_index     → primary key only")
print("   • events_with_indexes → primary key + 5 extra indexes (incl. GIN on JSONB)")


In [ ]:
print("🔧 Index Impact on Write Performance")
print("=" * 60)

# Why generate the rows *inside* PostgreSQL instead of looping in Python?
# A Python loop pays one network round-trip per row. That cost is identical
# for both tables, so it swamps the thing we are trying to measure -- run this
# benchmark row-at-a-time and index overhead shows up as ~2%, which is noise.
# `INSERT ... SELECT FROM generate_series(...)` is a single round-trip, so what
# we time is almost entirely server-side work: heap insert + index maintenance.
# That is the signal this section is about.
#
# The payload varies per row on purpose: a GIN index over 50,000 identical
# JSONB documents is cheap, over 50,000 distinct ones it is not.
num_rows = 50_000
trials = 3

# NOTE: the SQL below contains `%` as PostgreSQL's modulo operator, so it must
# NOT be handed to psycopg2 with a parameter tuple -- psycopg2 would try to read
# `% 1000` as a placeholder. The row count is an int we control, so we format it
# into the string and call execute() with no params (which skips interpolation).
INSERT_SQL = """
    INSERT INTO {table} (event_type, user_id, payload)
    SELECT 'benchmark',
           i % 1000,
           jsonb_build_object('test', true, 'i', i, 'tag', 'tag_' || (i % 97))
    FROM generate_series(1, {rows}) AS i
"""


def avg_time(table: str) -> float:
    times = []
    for _ in range(trials):
        c = get_connection()
        cur = c.cursor()
        cur.execute(f"TRUNCATE {table}")
        c.commit()
        start = time.time()
        cur.execute(INSERT_SQL.format(table=table, rows=num_rows))
        c.commit()
        times.append(time.time() - start)
        c.close()
    return sum(times) / len(times)


print(f"\nInserting {num_rows:,} rows server-side, averaged over {trials} trials...\n")

time_no_index = avg_time("events_no_index")
print("📊 Table WITHOUT extra indexes (PK only):")
print(f"   Time: {time_no_index:.2f}s")
print(f"   Rate: {num_rows / time_no_index:,.0f} inserts/sec")

time_with_index = avg_time("events_with_indexes")
print("\n📊 Table WITH 5 extra indexes (incl. GIN on JSONB):")
print(f"   Time: {time_with_index:.2f}s")
print(f"   Rate: {num_rows / time_with_index:,.0f} inserts/sec")

slowdown = ((time_with_index - time_no_index) / time_no_index) * 100
print(f"\n💡 Indexes added {slowdown:+.1f}% overhead to writes.")
print("   Every INSERT must update every index → the more indexes,")
print("   the more pages PostgreSQL touches per row.")
print("   (GIN indexes on JSONB are especially expensive to maintain.)")

# If this fails, the benchmark has stopped measuring index maintenance --
# something else (client loop, network, disk) became the bottleneck again and
# the section no longer demonstrates its own point.
assert slowdown > 20, (
    f"expected >20% write overhead from 5 indexes, measured {slowdown:+.1f}% -- "
    f"this benchmark is being dominated by something other than index maintenance"
)


## 📦 Bulk Insert Techniques

In [ ]:
print("📦 Bulk Insert Comparison")
print("=" * 60)

conn = get_connection()
cursor = conn.cursor()
cursor.execute("TRUNCATE events_no_index")
conn.commit()
conn.close()

num_rows = 5000
data = [('bulk_test', i % 1000, '{"batch": true}') for i in range(num_rows)]

print(f"\nInserting {num_rows} rows with different methods...")

conn = get_connection()
cursor = conn.cursor()
start = time.time()
for row in data:
    cursor.execute(
        "INSERT INTO events_no_index (event_type, user_id, payload) VALUES (%s, %s, %s)",
        row
    )
conn.commit()
time_individual = time.time() - start
cursor.execute("TRUNCATE events_no_index")
conn.commit()
conn.close()

print(f"\n1️⃣ Individual INSERTs:")
print(f"   Time: {time_individual:.2f}s")
print(f"   Rate: {num_rows/time_individual:.0f} inserts/sec")

In [ ]:
conn = get_connection()
cursor = conn.cursor()
start = time.time()
psycopg2.extras.execute_batch(
    cursor,
    "INSERT INTO events_no_index (event_type, user_id, payload) VALUES (%s, %s, %s)",
    data,
    page_size=100
)
conn.commit()
time_batch = time.time() - start
cursor.execute("TRUNCATE events_no_index")
conn.commit()
conn.close()

print(f"\n2️⃣ execute_batch (page_size=100):")
print(f"   Time: {time_batch:.2f}s")
print(f"   Rate: {num_rows/time_batch:.0f} inserts/sec")
print(f"   Speedup: {time_individual/time_batch:.1f}x faster")

In [ ]:
from io import StringIO

conn = get_connection()
cursor = conn.cursor()

csv_data = StringIO()
for row in data:
    csv_data.write(f"{row[0]}\t{row[1]}\t{row[2]}\n")
csv_data.seek(0)

start = time.time()
cursor.copy_from(
    csv_data,
    'events_no_index',
    columns=('event_type', 'user_id', 'payload')
)
conn.commit()
time_copy = time.time() - start
conn.close()

print(f"\n3️⃣ COPY (bulk load):")
print(f"   Time: {time_copy:.2f}s")
print(f"   Rate: {num_rows/time_copy:.0f} inserts/sec")
print(f"   Speedup: {time_individual/time_copy:.1f}x faster")

print("\n" + "=" * 60)
print("📊 Summary:")
print(f"   Individual:    {num_rows/time_individual:>8.0f} rows/sec")
print(f"   Batch:         {num_rows/time_batch:>8.0f} rows/sec")
print(f"   COPY:          {num_rows/time_copy:>8.0f} rows/sec")
print(f"\n💡 On this run COPY was {time_individual/time_copy:.0f}x faster than")
print(f"   row-at-a-time INSERTs (the usual range is 10-100x, depending on")
print(f"   how much of the gap is network round-trips vs SQL parsing).")

# The ladder has to stay a ladder. If COPY ever loses to execute_batch, or
# execute_batch loses to row-at-a-time, this section is no longer teaching
# what it claims.
assert time_batch < time_individual, (
    f"expected execute_batch to beat row-at-a-time, got "
    f"{time_batch:.3f}s vs {time_individual:.3f}s"
)
assert time_copy < time_batch, (
    f"expected COPY to beat execute_batch, got {time_copy:.3f}s vs {time_batch:.3f}s"
)


## 🪵 UNLOGGED Tables: Skip the Write-Ahead Log

Every `INSERT` in PostgreSQL is first written to the **WAL (Write-Ahead Log)** so the
database can recover after a crash. That durability has a cost: each committed row
must be flushed (`fsync`) to disk.

For **staging tables, ETL scratch, or analytics ingestion**, you often don't need
crash safety — if the loader fails, you just rerun it. PostgreSQL lets you opt out
with an `UNLOGGED` table, which skips WAL writes entirely.

> ⚠️ UNLOGGED tables are **truncated on crash** and are **not replicated**.
> Never use them for data you can't regenerate.

> ⚠️ Measured on Docker Desktop (macOS/Windows), the gap below is often small:
> the VM layer buffers `fsync`, so "durable" is already cheap and there is less
> for `UNLOGGED` to skip. On a Linux host with a real durable `fsync` the same
> benchmark typically shows 2-5x. Take the *direction* from this cell, not the
> magnitude.


In [ ]:
print("🪵 UNLOGGED vs LOGGED table inserts")
print("=" * 60)

conn = get_connection()
cursor = conn.cursor()
cursor.execute("DROP TABLE IF EXISTS events_unlogged")
cursor.execute("""
    CREATE UNLOGGED TABLE events_unlogged (
        id SERIAL PRIMARY KEY,
        event_type VARCHAR(50),
        user_id INTEGER,
        payload JSONB,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
""")
conn.commit()
conn.close()

# To actually see the cost of the WAL, we have to commit often.
# One big batch inside a single transaction only fsyncs once, so the WAL
# overhead gets hidden. Real apps commit every row (or every few rows),
# which is exactly when UNLOGGED starts to shine.
rows = 1000

def timed_commit_per_row(table):
    c = get_connection()
    c.autocommit = True   # fsync after every INSERT
    cur = c.cursor()
    cur.execute(f"TRUNCATE {table}")
    start = time.time()
    for i in range(rows):
        cur.execute(
            f"INSERT INTO {table} (event_type, user_id, payload) VALUES (%s, %s, %s)",
            ("bench", i % 1000, '{"x":1}'),
        )
    elapsed = time.time() - start
    c.close()
    return elapsed

t_logged = timed_commit_per_row("events_no_index")
t_unlogged = timed_commit_per_row("events_unlogged")

print(f"\n{rows} INSERTs with commit-per-row (each row fsyncs the WAL):")
print(f"   LOGGED   (WAL on):  {t_logged:.2f}s  ->  {rows/t_logged:,.0f} rows/sec")
print(f"   UNLOGGED (WAL off): {t_unlogged:.2f}s  ->  {rows/t_unlogged:,.0f} rows/sec")
print(f"\n💡 Speedup from skipping WAL: {t_logged / max(t_unlogged, 1e-9):.1f}x")
print("   The win grows with commit frequency: high-frequency single-row")
print("   inserts are where UNLOGGED (or batched commits) help the most.")
print("   Trade-off: UNLOGGED data is lost if PostgreSQL crashes.")

# We deliberately do NOT assert a speedup here -- see the note above, a
# containerised fsync can be almost free. What must always hold is the
# direction: skipping the WAL cannot make inserts materially *slower*.
assert t_unlogged <= t_logged * 1.2, (
    f"UNLOGGED should never be materially slower than LOGGED, got "
    f"{t_unlogged:.2f}s vs {t_logged:.2f}s"
)


## 🚀 `execute_values`: The Fastest Pure-INSERT Path

`execute_batch` sends each row as a separate `INSERT`. `execute_values`
(also from `psycopg2.extras`) builds **one** multi-row `INSERT` statement,
which typically runs 2–5× faster than `execute_batch` and is easier for
PostgreSQL to plan.

Think of the ladder as:

```
 slowest  ->  executemany()       one INSERT per row
           ->  execute_batch()    many INSERTs in one round-trip
           ->  execute_values()   one INSERT with many VALUES (...)
 fastest  ->  COPY                bulk stream, no SQL parsing per row
```


In [ ]:
print("🚀 execute_values vs execute_batch")
print("=" * 60)

conn = get_connection()
cursor = conn.cursor()
cursor.execute("TRUNCATE events_no_index")
conn.commit()

rows = 5000
data2 = [("ev_test", i % 1000, '{"m":true}') for i in range(rows)]

start = time.time()
psycopg2.extras.execute_values(
    cursor,
    "INSERT INTO events_no_index (event_type, user_id, payload) VALUES %s",
    data2,
    page_size=500,
)
conn.commit()
t_values = time.time() - start
cursor.execute("TRUNCATE events_no_index")
conn.commit()
conn.close()

print(f"\nexecute_values: {t_values:.2f}s  ->  {rows/t_values:,.0f} rows/sec")
print(f"execute_batch:  {time_batch:.2f}s  ->  {num_rows/time_batch:,.0f} rows/sec"
      f"   (measured two cells up, same 5,000 rows)")
print(f"\n💡 execute_values was {time_batch/t_values:.1f}x faster than execute_batch here.")
print("   For INSERTs, prefer execute_values over execute_batch.")
print("   For multi-million-row loads, prefer COPY (shown above).")

# One statement with many VALUES must beat many statements in one round-trip.
# If it does not, the ladder in the markdown above is wrong.
assert t_values < time_batch, (
    f"expected execute_values to beat execute_batch, got "
    f"{t_values:.3f}s vs {time_batch:.3f}s"
)


## 🎛️ Write Optimization Strategies

In [ ]:
print("🎛️ PostgreSQL Write Optimization Strategies")
print("=" * 60)
print("""
1. REDUCE INDEX OVERHEAD
─────────────────────────────────────────────────────────────
   • Drop unnecessary indexes
   • Use partial indexes (WHERE clause)
   • Defer index creation for bulk loads
   
   -- Drop during bulk load
   DROP INDEX idx_events_type;
   -- ... bulk insert ...
   CREATE INDEX idx_events_type ON events(event_type);

2. BATCH COMMITS
─────────────────────────────────────────────────────────────
   • Commit every N rows instead of every row
   • Trade durability for speed
   
   -- Instead of commit after each INSERT
   BEGIN;
   INSERT ...; INSERT ...; INSERT ...;  -- 1000 rows
   COMMIT;

3. DISABLE CONSTRAINTS TEMPORARILY
─────────────────────────────────────────────────────────────
   • Foreign key checks have overhead
   • Disable during trusted bulk loads
   
   SET session_replication_role = 'replica';  -- Disable FK
   -- ... bulk insert ...
   SET session_replication_role = 'origin';   -- Re-enable

4. TUNE WAL SETTINGS
─────────────────────────────────────────────────────────────
   • synchronous_commit = off (risky but fast)
   • wal_buffers = larger value
   • checkpoint_completion_target = 0.9
""")

## 🧪 Quick Quiz

1. **Why is Cassandra faster for writes than PostgreSQL?**

2. **When should you drop indexes before bulk loading?**

3. **What's the trade-off with `synchronous_commit = off`?**

In [ ]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Cassandra's write advantage:")
print("   - Append-only (sequential writes)")
print("   - No in-place updates (no seeks)")
print("   - Compaction happens in background")
print("   - Trade-off: Slower reads, eventual consistency")
print()
print("2. When to drop indexes:")
print("   - Large bulk loads (>100k rows)")
print("   - When you control all the data")
print("   - Recreate index after load (faster!)")
print()
print("3. synchronous_commit = off:")
print("   - Writes return before WAL flush")
print("   - Risk: Lose last ~200ms of data on crash")
print("   - Use for: Analytics, logs (not transactions!)")

## 📚 Summary

### Key Takeaways

1. **Choose the right database** - Log-structured for write-heavy
2. **Indexes hurt writes** - Only index what you query
3. **Bulk > Individual** - COPY is 10-100x faster
4. **Batch commits** - Reduce transaction overhead
5. **Know your trade-offs** - Speed vs durability

### Next Up

In **Notebook 3**, we'll learn sharding and partitioning:
- Horizontal sharding strategies
- Choosing partition keys
- Avoiding hot spots